# 🎮 Deep Q-Network (DQN) — Reinforcement Learning mit PyTorch

**CartPole-v1 mit Deep Q-Learning lösen**

In diesem Notebook lernst du:
- **Reinforcement Learning**: Agent interagiert mit Environment
- **DQN**: Q-Funktion mit neuronalem Netz approximieren
- **Experience Replay**: Vergangene Erfahrungen wiederverwenden
- **Target Network**: Stabilisiert das Training
- **Epsilon-Greedy**: Exploration vs. Exploitation
- Visualisierung: Reward-Kurve, Epsilon-Decay, Beispiel-Trajektorie

In [ ]:
import random
from collections import deque
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn, optim

# Reproduzierbarkeit
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

## 1. Environment: CartPole-v1

**Ziel:** Einen Stab auf einem Wagen balancieren.
- **State (4D):** Position, Geschwindigkeit, Winkel, Winkelgeschwindigkeit
- **Actions (2):** Links (0) oder Rechts (1) schieben
- **Reward:** +1 pro Zeitschritt
- **Gelöst:** Durchschnittlich 195+ Reward über 100 Episoden

In [ ]:
try:
    import gymnasium as gym
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gymnasium"])
    import gymnasium as gym

env = gym.make("CartPole-v1")
STATE_DIM = env.observation_space.shape[0]   # 4
ACTION_DIM = env.action_space.n              # 2

print(f"Environment: CartPole-v1")
print(f"State-Dimension: {STATE_DIM}")
print(f"Action-Dimension: {ACTION_DIM}")

## 2. Hyperparameter

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPISODES = 500
BATCH_SIZE = 64
GAMMA = 0.99                # Discount-Faktor
EPSILON_START = 1.0         # Start: 100% Exploration
EPSILON_END = 0.01          # Minimum: 1% Exploration
EPSILON_DECAY = 0.995       # Decay pro Episode
LEARNING_RATE = 0.001
TARGET_UPDATE = 10          # Target-Network Update alle N Episoden
MEMORY_SIZE = 10_000        # Replay-Buffer-Größe
MIN_MEMORY = 1_000          # Mindest-Erfahrungen vor Training

print(f"Device: {DEVICE}")

## 3. Q-Network

Ein einfaches Feed-Forward-Netz, das Q-Werte für jede Action schätzt:
- **Input:** State (4D)
- **Output:** Q-Wert für jede Action (2D)

Q(s, a) = erwarteter zukünftiger Reward, wenn wir in State s Action a wählen.

In [ ]:
class QNetwork(nn.Module):
    """Einfaches Feed-Forward-Netz für Q-Wert-Approximation."""
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, x):
        return self.net(x)

print(QNetwork(STATE_DIM, ACTION_DIM))

## 4. Replay Buffer

Speichert Erfahrungen `(state, action, reward, next_state, done)` und sampelt zufällig Batches.

**Warum?**
- Bricht Korrelation zwischen aufeinanderfolgenden Samples
- Wiederverwendung seltener Erfahrungen
- Stabilisiert das Training

In [ ]:
class ReplayBuffer:
    """Experience Replay Buffer mit zufälligem Sampling."""
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)).to(DEVICE),
            torch.LongTensor(actions).to(DEVICE),
            torch.FloatTensor(rewards).to(DEVICE),
            torch.FloatTensor(np.array(next_states)).to(DEVICE),
            torch.FloatTensor(dones).to(DEVICE),
        )

    def __len__(self):
        return len(self.buffer)

## 5. DQN-Agent

Der komplette Agent mit:
- **Policy Network**: Wählt Actions (wird trainiert)
- **Target Network**: Stabiler Q-Wert-Schätzer (periodisch aktualisiert)
- **Epsilon-Greedy**: Mit Wahrscheinlichkeit ε zufällige Action, sonst beste

In [ ]:
class DQNAgent:
    """DQN-Agent mit Target-Network und Epsilon-Greedy."""
    def __init__(self, state_dim, action_dim):
        self.action_dim = action_dim
        self.policy_net = QNetwork(state_dim, action_dim).to(DEVICE)
        self.target_net = QNetwork(state_dim, action_dim).to(DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()  # Target-Network wird nie direkt trainiert

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=LEARNING_RATE)
        self.memory = ReplayBuffer(MEMORY_SIZE)
        self.epsilon = EPSILON_START

    def select_action(self, state, evaluate=False):
        """Epsilon-Greedy Action-Selektion."""
        if evaluate or random.random() > self.epsilon:
            # Greedy: Beste Action nach Q-Network
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
                q_values = self.policy_net(state_t)
                return q_values.argmax(dim=1).item()
        else:
            # Exploration: Zufällige Action
            return random.randrange(self.action_dim)

    def update(self):
        """Ein Trainingsschritt mit Experience Replay."""
        if len(self.memory) < MIN_MEMORY:
            return None

        states, actions, rewards, next_states, dones = self.memory.sample(BATCH_SIZE)

        # Current Q values: Q(s, a)
        q_values = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Target Q values (Double DQN)
        with torch.no_grad():
            # Policy-Network wählt beste Action...
            next_actions = self.policy_net(next_states).argmax(dim=1)
            # ...Target-Network bewertet sie
            next_q_values = self.target_net(next_states).gather(1, next_actions.unsqueeze(1)).squeeze(1)
            target_q_values = rewards + GAMMA * next_q_values * (1 - dones)

        # MSE-Loss
        loss = nn.MSELoss()(q_values, target_q_values)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)  # Stabilität
        self.optimizer.step()

        return loss.item()

    def update_target(self):
        """Target-Network = Policy-Network (Hard Update)."""
        self.target_net.load_state_dict(self.policy_net.state_dict())

    def decay_epsilon(self):
        self.epsilon = max(EPSILON_END, self.epsilon * EPSILON_DECAY)


agent = DQNAgent(STATE_DIM, ACTION_DIM)
print(f"Policy-Network Parameter: {sum(p.numel() for p in agent.policy_net.parameters()):,}")

## 6. Training

Der Trainingsloop:
1. **Episode starten**: Environment resetten
2. **Action wählen**: Epsilon-Greedy
3. **Erfahrung sammeln**: `(s, a, r, s', done)` in Replay Buffer
4. **Lernen**: Batch aus Buffer sampeln, Q-Network updaten
5. **Epsilon decay**: Exploration langsam reduzieren
6. **Target-Update**: Alle N Episoden Target-Network aktualisieren

In [ ]:
episode_rewards, epsilons, losses, moving_avg = [], [], [], []

print("── Training ──")
for episode in range(1, EPISODES + 1):
    state, _ = env.reset()
    episode_reward = 0.0
    episode_losses = []

    while True:
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        agent.memory.push(state, action, reward, next_state, done)
        state = next_state
        episode_reward += reward

        loss = agent.update()
        if loss is not None:
            episode_losses.append(loss)

        if done:
            break

    episode_rewards.append(episode_reward)
    epsilons.append(agent.epsilon)
    avg_loss = np.mean(episode_losses) if episode_losses else 0.0
    losses.append(avg_loss)

    agent.decay_epsilon()

    if episode % TARGET_UPDATE == 0:
        agent.update_target()

    # Gleitender Durchschnitt (letzte 100 Episoden)
    ma = np.mean(episode_rewards[-100:]) if len(episode_rewards) >= 100 else np.mean(episode_rewards)
    moving_avg.append(ma)

    if episode % 50 == 0 or episode == 1:
        print(f"Episode {episode:4d}/{EPISODES} | Reward: {episode_reward:6.1f} | "
              f"Avg100: {ma:6.1f} | Epsilon: {agent.epsilon:.3f} | Loss: {avg_loss:.4f}")

env.close()

print(f"\nFinaler Avg100-Reward: {moving_avg[-1]:.2f}")
solved = moving_avg[-1] >= 195.0
print(f"CartPole {'GELÖST! 🎉' if solved else 'noch nicht gelöst (Ziel: 195.0)'}")

## 7. Visualisierung

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Reward pro Episode
axes[0, 0].plot(episode_rewards, alpha=0.4, color="blue", linewidth=0.8, label="Episode Reward")
axes[0, 0].plot(moving_avg, color="red", linewidth=2, label="Moving Avg (100)")
axes[0, 0].axhline(y=195.0, color="green", linestyle="--", label="Solved (195)")
axes[0, 0].set_title("Reward pro Episode")
axes[0, 0].set_xlabel("Episode")
axes[0, 0].set_ylabel("Reward")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Epsilon-Decay
axes[0, 1].plot(epsilons, color="purple", linewidth=2)
axes[0, 1].set_title("Epsilon-Decay (Exploration → Exploitation)")
axes[0, 1].set_xlabel("Episode")
axes[0, 1].set_ylabel("Epsilon")
axes[0, 1].grid(True, alpha=0.3)

# Loss
axes[1, 0].plot(losses, color="orange", alpha=0.7, linewidth=1)
axes[1, 0].set_title("Training Loss")
axes[1, 0].set_xlabel("Episode")
axes[1, 0].set_ylabel("Loss")
axes[1, 0].grid(True, alpha=0.3)

# Reward-Histogramm
axes[1, 1].hist(episode_rewards, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
axes[1, 1].axvline(x=195.0, color="green", linestyle="--", linewidth=2, label="Solved (195)")
axes[1, 1].set_title("Reward-Verteilung")
axes[1, 1].set_xlabel("Reward")
axes[1, 1].set_ylabel("Häufigkeit")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

### Beispiel-Trajektorie (Evaluation)

Jetzt spielen wir eine Episode **ohne Exploration** (rein greedy) und zeigen Screenshots.

In [ ]:
print("── Evaluation: Beispiel-Trajektorie ──")
eval_env = gym.make("CartPole-v1", render_mode="rgb_array")
state, _ = eval_env.reset()
total_reward, step_count = 0.0, 0
frames = []

for step in range(500):
    action = agent.select_action(state, evaluate=True)  # Kein Epsilon!
    state, reward, terminated, truncated, _ = eval_env.step(action)
    total_reward += reward
    step_count += 1
    frames.append(eval_env.render())
    if terminated or truncated:
        break

eval_env.close()
print(f"Evaluations-Reward: {total_reward:.1f} in {step_count} Schritten")

# Zeige 4 Frames
fig3, axes3 = plt.subplots(1, 4, figsize=(16, 4))
indices = np.linspace(0, len(frames) - 1, 4, dtype=int)
for i, idx in enumerate(indices):
    axes3[i].imshow(frames[idx])
    axes3[i].set_title(f"Step {idx}")
    axes3[i].axis("off")
fig3.suptitle(f"Beispiel-Trajektorie (Reward: {total_reward:.1f})", fontsize=13)
fig3.tight_layout()
plt.show()

## Zusammenfassung

| Konzept | Beschreibung |
|---------|-------------|
| **DQN** | Q-Funktion mit Deep Learning approximieren |
| **Experience Replay** | Zufällige Batches aus vergangenen Erfahrungen |
| **Target Network** | Stabiler Q-Wert-Schätzer, periodisch aktualisiert |
| **Epsilon-Greedy** | ε = Exploration, 1-ε = Exploitation |
| **Double DQN** | Policy-Network wählt Action, Target-Network bewertet |
| **Gradient Clipping** | Verhindert explodierende Gradienten |

**DQN-Algorithmus:**
1. Wähle Action a mit Epsilon-Greedy
2. Führe a aus, erhalte Reward r und neuen State s'
3. Speichere (s, a, r, s') im Replay Buffer
4. Sample zufälligen Batch, berechne Target: r + γ·max Q(s', a')
5. Minimiere MSE zwischen Q(s, a) und Target